# Structured Request Reliability

One notebook for verifying frozen adapters, inspecting reference results, running local regression checks, and exercising the API. The implementation lives in `src/request_reliability/` and is shared with the CLI.

**Start in replay mode.** Run All verifies files, recomputes saved scores and displays existing results. It does not load a model or start missing experiments. Change `MODE` to `"live"` for new inference. The existing LoRA memory stops remain visible.

The reference experiment used CUDA. Existing MPS QLoRA evidence covers eight requests: all eight outputs match, but only four are entirely correct. Output agreement and task accuracy are different measurements.

## 1. Locate this project
Import standard-library helpers. No model packages or weights are loaded.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from uuid import uuid4

Choose the standalone project directory. Automatic discovery works when the kernel starts in the project or its `notebooks/` folder. Set `PROJECT_ROOT_OVERRIDE` to the absolute path of your copy if necessary.

In [ ]:
PROJECT_ROOT_OVERRIDE = None  # Example: Path("/absolute/path/to/structured-request-reliability")

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = (Path.cwd(), *Path.cwd().parents)
    PROJECT_ROOT = next(
        (
            p
            for p in candidates
            if (p / "src/request_reliability/inference.py").is_file()
            and (p / "data/frozen/manifest.json").is_file()
        ),
        None,
    )
    if PROJECT_ROOT is None:
        raise RuntimeError("Set PROJECT_ROOT_OVERRIDE to this project's directory.")
if not (PROJECT_ROOT / "data/frozen/manifest.json").is_file():
    raise RuntimeError("Copy this project's complete private data/frozen bundle before running.")
os.chdir(PROJECT_ROOT)
print("Project:", PROJECT_ROOT)
print("Kernel Python:", sys.executable)

## 2. Dependencies
The prepared `.venv` already contains the notebook dependencies. On another kernel, enable this cell once to install the hashed requirements into that kernel’s Python with pip. Restart the kernel after installation, disable the switch, and continue. This cell does not download model weights.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--require-hashes",
            "-r",
            str(PROJECT_ROOT / "requirements.txt"),
        ]
    )
    raise RuntimeError(
        "Dependencies installed. Restart the kernel and set INSTALL_DEPENDENCIES=False."
    )
print("Using current kernel packages; dependency installation skipped.")

Use this project’s implementation and isolate its caches before importing model libraries. Imports never run inference. Use Python 3.13 and the prepared environment for the verified local configuration.

In [ ]:
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Select this project's Python 3.13 environment.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["MPLCONFIGDIR"] = str(PROJECT_ROOT / ".cache/matplotlib")
# Import after selecting this project's source path.
from request_reliability.inference import configure_offline  # noqa: E402

configure_offline(PROJECT_ROOT)

# Configure local caches before importing runtime/display libraries.
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402
from IPython.display import JSON, Markdown, display  # noqa: E402

from request_reliability.workflow import NotebookConfig, NotebookSession  # noqa: E402

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## 3. Choose execution settings
`replay` reads saved local evidence and never launches missing runs. `live` generates new results in unique report directories. `smoke8` selects the first eight test requests in their original order; `full` explicitly selects all 2,974 reused test requests.

QLoRA is the default live path because it passed the bounded MPS check. Enable LoRA only after memory conditions improve: dense models require at least 6 GiB available unified memory; NF4 requires 3 GiB. These checks do not guarantee fit. The swap guard remains enabled.

In [ ]:
MODE = "replay"  # "replay" or "live"
PRESET = "smoke8"  # "smoke8" or "full"
DEVICE = "auto"  # "auto", "mps", "cuda", "cpu"
RUN_LORA = False  # Explicitly enable only after resolving memory pressure.
RUN_BASELINES = False  # Optional live base and NF4-base comparisons.
RUN_BENCHMARKS = False  # Explicit live timing; never inferred from old CUDA timings.
RUN_API_CHECK = False  # Scoped real-model ASGI check, not a persistent server.
WRITE_SESSION_REPORT = False  # True writes a new report; existing evidence is preserved.
PREDICTION_SYSTEM = "qlora"
PREDICTION_REQUEST = "wake me up at five am this week"

config = NotebookConfig(
    mode=MODE,
    device=DEVICE,
    preset=PRESET,
    timeout_seconds=60,
    benchmark_requests=8,
    benchmark_repetitions=3,
)
session = NotebookSession(PROJECT_ROOT, config=config)
display(JSON(config.to_dict()))
print("Cases per enabled comparison:", session.actual_count)

## 4. Inspect hardware
Inspect available devices and memory without loading weights. Missing telemetry is shown as unavailable. Automatic selection prefers MPS on this Mac, then CUDA, then CPU. CPU is a distinct FP32 execution condition.

In [ ]:
hardware = session.hardware()
display(JSON(hardware))

## 5. Verify the frozen inputs
Hash-check all copied inputs. Normal operation uses only this project’s files. The original training project is never opened by this notebook.

In [ ]:
integrity = session.verify(full=True)
display(JSON(integrity))

Inspect the bundle inventory. The final epoch-five adapters, matching backbones, tokenizer, prompt, ontology and evaluation examples are all versioned together.

In [ ]:
inventory = pd.DataFrame(session.inventory())
inventory["MiB"] = inventory["bytes"] / 1024**2
display(inventory[["artifact", "MiB", "sha256"]])

## 6. Recompute the original scores
Re-score the saved raw answers from all four systems. This verifies the scoring implementation and imported evidence; it does not generate new answers or measure Mac performance.

In [ ]:
reference_audit = session.reference_audit(recompute=True, save=False)
reference_rows = []
for system, result in reference_audit["systems"].items():
    metrics = result["metrics"]
    reference_rows.append(
        {
            "system": system,
            "whole_record_correct_%": 100 * metrics["joint_exact_match"],
            "schema_valid_%": 100 * metrics["schema_valid_rate"],
            "intent_correct_%": 100 * metrics["intent_accuracy"],
            "slot_F1": metrics["slot_f1"],
        }
    )
reference_table = pd.DataFrame(reference_rows).set_index("system")
print("Recomputed original responses per system:", reference_audit["example_count"])
display(reference_table.round(3))

Compare whole-response correctness with valid JSON structure in the original CUDA experiment. A valid structure alone does not establish a correct intent or correct slot values.

In [ ]:
plot_table = reference_table[["whole_record_correct_%", "schema_valid_%"]]
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
plot_table.plot.bar(ax=ax, color=["#236B8E", "#E5AA50"], width=0.72)
ax.set(
    title="Original CUDA reference — 2,974 requests per system", ylabel="Requests (%)", xlabel=""
)
ax.set_ylim(0, 113)
ax.tick_params(axis="x", rotation=0)
ax.legend(["Entire response correct", "Schema valid"], loc="upper left", frameon=False)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=9, padding=3)
plt.show()

## 7. Inspect existing local status
Saved MPS results are separate from the full CUDA reference. Failures and missing work remain visible. A cached result is not a new execution.

In [ ]:
display(pd.DataFrame(session.status_table()))
display(pd.DataFrame(session.comparison_table()))

## 8. QLoRA regression check
In replay mode this displays matching evidence for the configured scope, or a missing result. In live mode it reloads the frozen QLoRA adapter, verifies prompt tokens, generates the configured cases, closes the model and writes a new result directory. A returned failed result remains a failure.

In [ ]:
qlora_comparison = session.compare("qlora")
display(JSON(qlora_comparison))

## 9. LoRA regression check
Replay displays the saved loading failure. Live execution requires `RUN_LORA=True` and sufficient memory. No automatic retry follows a memory failure.

In [ ]:
if MODE == "replay" or RUN_LORA:
    lora_comparison = session.compare("lora")
else:
    lora_comparison = {
        "status": "skipped",
        "reason": "Set RUN_LORA=True after resolving memory pressure.",
        "inference_performed": False,
    }
display(JSON(lora_comparison))

## 10. Optional unadapted baselines
The frozen base and NF4-base models are available for the same comparison. They are optional live work. Missing local baseline results remain missing in replay mode; the original CUDA scores are not substituted.

In [ ]:
baseline_comparisons = {}
if MODE == "replay" or RUN_BASELINES:
    for system in ("base", "quantized_base"):
        baseline_comparisons[system] = session.compare(system)
else:
    baseline_comparisons = {"status": "skipped", "reason": "RUN_BASELINES=False"}
display(JSON(baseline_comparisons))

## 11. Inspect one prediction
Edit `PREDICTION_REQUEST` in the settings cell. Replay can show only an existing local prediction for that exact request. Live mode loads the selected model for this request and releases it afterward. Schema validity and semantic correctness remain separate.

In [ ]:
prediction = session.predict(PREDICTION_SYSTEM, PREDICTION_REQUEST)
display(JSON(prediction))

## 12. Benchmark loading and response time
A live benchmark separates loading, first request, excluded warmups and measured validation requests. Memory values are point samples, not peaks. Existing timing from the original CUDA experiment is not a local benchmark.

In [ ]:
if MODE == "replay" or RUN_BENCHMARKS:
    benchmark = session.benchmark(PREDICTION_SYSTEM)
else:
    benchmark = {
        "status": "skipped",
        "reason": "RUN_BENCHMARKS=False",
        "inference_performed": False,
    }
display(JSON(benchmark))

## 13. Check the API in this notebook
The live check uses the real selected model through an in-process ASGI client: a valid request, repeat request, invalid input and clean shutdown. It does not start a TCP listener. The asynchronous scope avoids Jupyter event-loop conflicts and retains model ownership until in-flight work ends.

In [ ]:
api_check = {
    "status": "missing" if MODE == "replay" else "skipped",
    "reason": "No matching saved API check; enable live mode and RUN_API_CHECK for new inference.",
    "inference_performed": False,
}
if MODE == "live" and RUN_API_CHECK:
    from request_reliability.cli import save_json
    from request_reliability.notebook_api import check_api

    api_path = session.results_dir() / f"api-check-{uuid4().hex[:8]}.json"
    try:
        api_check = await check_api(
            session.bundle,
            system=PREDICTION_SYSTEM,
            device=DEVICE,
            timeout_seconds=config.timeout_seconds,
        )
        api_check = {**api_check, "inference_performed": True}
    except Exception as exc:
        api_check = {
            "status": "failed",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "inference_requested": True,
            "inference_performed": None,
        }
    api_check = {**api_check, "evidence_path": str(api_path)}
    save_json(api_path, api_check)
elif MODE == "replay":
    api_candidates = list((PROJECT_ROOT / "reports").rglob("api-check-*.json"))
    api_candidates.append(PROJECT_ROOT / "reports/local-api-check.json")
    api_candidates = sorted(
        (path for path in api_candidates if path.is_file()),
        key=lambda path: path.stat().st_mtime_ns,
        reverse=True,
    )
    for api_path in api_candidates:
        saved_api = json.loads(api_path.read_text())
        identity = saved_api.get("identity", {})
        if identity.get("system") == PREDICTION_SYSTEM and (
            DEVICE == "auto" or identity.get("device") == DEVICE
        ):
            api_check = {
                **saved_api,
                "recorded_status": saved_api.get("status"),
                "status": "cached",
                "evidence_path": str(api_path),
                "inference_performed": False,
            }
            break
display(JSON(api_check))

## 14. Review or save this session
The report identifies cached, newly executed, failed, skipped and missing work. Saving creates a fresh session directory and records implementation/configuration hashes. Existing reports and frozen data remain untouched.

In [ ]:
session_results = {
    "qlora": qlora_comparison,
    "lora": lora_comparison,
    **(
        {f"baseline_{name}": value for name, value in baseline_comparisons.items()}
        if "status" not in baseline_comparisons
        else {"baselines": baseline_comparisons}
    ),
    "prediction": prediction,
    "benchmark": benchmark,
    "api": api_check,
}
session_report = session.report(session_results, write=WRITE_SESSION_REPORT)
display(Markdown(session_report["markdown"]))
if session_report["path"] is not None:
    print("New report:", session_report["path"])

## 15. Serve from a terminal
For a persistent local server, use the command below in a terminal after notebook inference has finished. A second server would load a second model. This cell prints the command only; it does not launch anything.

In [ ]:
import shlex

server_command = [
    sys.executable,
    "-m",
    "request_reliability.cli",
    "--bundle",
    str(session.bundle),
    "serve",
    "--system",
    PREDICTION_SYSTEM,
    "--device",
    DEVICE,
    "--port",
    "8000",
]
print("Working directory:", PROJECT_ROOT)
print(shlex.join(server_command))

## Interpretation and next action

- Matching eight outputs establishes a small regression check, not full-test equivalence or 100% task accuracy.
- The full test set has already been inspected; reuse here is regression evaluation, not untouched generalization evidence.
- Keep failed runs. Do not tune prompts, checkpoints or acceptance thresholds against this replay and relabel it a fresh test.
- Complete the fixed LoRA check only after memory conditions improve. Then deliberately select the wider regression and validation benchmark scope.
- Current native evidence and its limitations are in [the technical report](../../reports/technical-report.md). The [README](../../README.md) describes setup, data transfer, CLI usage and API behavior.